# What build10 added

Three additions, one theme: **the mechanism is not the surface**. Each
one takes something a program could already do the hard way and gives it
a name — while the rules underneath stay exactly where they were.

Everything here runs against whichever runtime the **Runtime** dropdown
names. These calls need `v5.5.1_build10` or newer; on an older one the
cells say so rather than failing strangely.


In [ ]:
-- Which build is under this notebook?
print(_VERSION)
print(("host library: %s"):format(type(host) == "table" and "present" or "absent"))
if type(host) == "table" then
  local names = {}
  for _, k in ipairs({ "spawn", "children", "events", "call" }) do
    names[#names + 1] = ("%s=%s"):format(k, type(host[k]))
  end
  print("  " .. table.concat(names, "  "))
  if type(host.spawn) ~= "function" then
    print("  (this runtime predates build10; pick a newer one in Runtime)")
  end
end


## 1. A supervisor that does not spell out the mechanism

`notebooks/swarm.ipynb` starts a child the way the runtime has always
allowed: declare `system/lifecycle` by its reserved name, push an op
table, then watch `system/events` and correlate what comes back. Every
line of that is real and none of it is *interesting* — it is the same
prologue in every supervisor anyone writes.

`host.spawn` is that prologue, once, in the library:

```lua
local kid = host.spawn{ code = src, caps = {...}, budget = {...} }
kid.kill()                 -- subtree kill, outcome awaited
host.children()            -- id -> handle, what is still alive
host.events(waitms)        -- {event, id, detail?}, the next child notice
```

The queues did not go away and are not hidden: they are the substrate,
and a program that wants them still has them. What changed is that
reaching for them is no longer the price of admission.


In [ ]:
-- The same shape as swarm.ipynb's raw version, through the library.
root_src = [[
local outbox = queue.lookup("outbox")

local worker = [==[
  local out = queue.lookup("outbox")
  out = out  -- a worker that simply exists, and exits
]==]

local kids = {}
for _, name in ipairs({ "alpha", "beta" }) do
  local kid = host.spawn{
    code   = "NAME = " .. string.format("%q", name) .. "\n" .. worker,
    caps   = { "queue:*" },
    budget = { instructions = 200000, memory_kb = 64 },
  }
  kids[#kids + 1] = kid.id
  queue.push(outbox, ("spawned %s as instance %d"):format(name, kid.id))
end

-- host.spawn already waited for each 'spawned' event, so by here the
-- children exist. What is left is hearing them finish.
local live = 0
for _ in pairs(host.children()) do live = live + 1 end
queue.push(outbox, ("host.children() holds %d"):format(live))

local seen = 0
while seen < #kids do
  local ev = host.events(2000)
  if not ev then break end
  if ev.event == "exited" or ev.event == "faulted" then
    seen = seen + 1
    queue.push(outbox, ("child %d %s"):format(ev.id, ev.event))
  end
end
]]

swarm.stop()
swarm.start{
  root = root_src,
  caps = { "lifecycle", "queue:*" },
  budget = { instructions = 5000000, memory_kb = 512 },
  max_instances = 8,
}
print("started")


In [ ]:
local evs = swarm.step(30)
print(#evs .. " event(s) the host saw:")
for _, e in ipairs(evs) do
  print(("  %-9s #%-3s %s"):format(e.event, tostring(e.id), tostring(e.detail or "")))
end

print()
print("what the supervisor said:")
for _, line in ipairs(swarm.drain("root", "outbox")) do
  print("  " .. tostring(line))
end


### What did *not* change

A denial still comes from the swarm, not from the library. A child can
only ever hold a subset of its parent's grants, and asking for more is
refused with the swarm's own sentence — `host.spawn` raises it rather
than inventing a second policy.


In [ ]:
swarm.stop()
swarm.start{
  root = [[
    local outbox = queue.lookup("outbox")
    local ok, err = pcall(host.spawn, {
      code = "local a = 1",
      caps = { "host:sql/exec" },   -- the parent does not hold this
    })
    queue.push(outbox, ok and "spawned (unexpected!)" or ("refused: " .. tostring(err)))
  ]],
  caps = { "lifecycle", "queue:*" },   -- note: no host:sql/exec here
  budget = { instructions = 2000000, memory_kb = 256 },
}
swarm.step(20)
for _, line in ipairs(swarm.drain("root", "outbox")) do print(line) end


## 2. A reply may set headers now

The listener has always carried `{conn, method, path, body}` in and
`{conn, status, body, content_type?}` back. Build10 adds a `headers` map
on the reply — redirects, caching directives, anything a real response
needs — gated the same way the request side has been since build7: the
deployment names an allowlist, and nothing else gets through.

The two directions differ in one deliberate way. Inbound, a bad header
is a lying *client*, and the host refuses the request to protect the
guest. Outbound, a bad header is a lying *guest*, and the party at risk
is the client — so that header is dropped **whole**, never truncated or
"cleaned", and the response still answers. A guest bug should not become
an outage.

Use the **Instances** panel's listener composer to drive this one: it
wires a listener, and a reply's allowlisted headers show up on the
exchange it completes.


## 3. Outbound HTTP, and the line it crosses

Releases now ship an outbound-HTTP capability, and this Lab answers the
same two calls in the page:

```lua
local r = host.call("rest/get", { url = "https://api.example.com/v1/thing" })
print(r.status, r.content_type, r.headers["x-rate-limit"], #r.body)
```

Every other connector here reaches something the page already owns — a
clock, a vendored database, a button someone pressed. **This one reaches
the network because the guest said so**, which is the line this Lab's
"no external requests" rule draws.

So it is off until a deployment wires it, and wiring it is not enough:
the grant has to say *where*.

```lua
connectors = { rest = { allow = { "https://api.example.com/v1/" } } }
```

A prefix, so the scheme and the path are part of what was granted. The
two cells below run entirely offline — they are the refusals, which are
the part worth seeing.


In [ ]:
-- Wired with no allowlist: a connector that reaches nothing, and says so.
swarm.stop()
swarm.start{
  root = [[
    local outbox = queue.lookup("outbox")
    local v, status, detail = host.try("rest/get", { url = "https://example.com/" })
    queue.push(outbox, ("status=%s"):format(tostring(status)))
    queue.push(outbox, tostring(detail))
  ]],
  caps = { "queue:*", "host:rest/get" },
  budget = { instructions = 2000000, memory_kb = 256 },
  connectors = { rest = {} },
}
swarm.step(20)
for _, line in ipairs(swarm.drain("root", "outbox")) do print(line) end


In [ ]:
-- Granted one prefix, asked for another.
swarm.stop()
swarm.start{
  root = [[
    local outbox = queue.lookup("outbox")
    local v, status, detail = host.try("rest/get", { url = "https://elsewhere.example/data" })
    queue.push(outbox, ("status=%s"):format(tostring(status)))
    queue.push(outbox, tostring(detail))
  ]],
  caps = { "queue:*", "host:rest/get" },
  budget = { instructions = 2000000, memory_kb = 256 },
  connectors = { rest = { allow = { "https://api.example.com/v1/" } } },
}
swarm.step(20)
for _, line in ipairs(swarm.drain("root", "outbox")) do print(line) end


### Two things worth knowing before you wire it

**The answer is deferred.** `fetch` cannot answer inside the step that
took the call, so the host *takes* the call, keeps stepping every other
instance, and delivers the reply when it lands. The guest sees an
ordinary parked hostcall; the swarm never stalls on one slow endpoint.
The C host reached the same design for the same reason.

**A page is subject to CORS, and no configuration here lifts it.** An
endpoint that does not allow this origin fails as an error the guest can
read, which is a real difference from a host that runs outside a browser.
If a call works from a terminal and not from this page, that is usually
what happened — and it is a fact about the endpoint, not about the
grant.
